# Five-class score prediction with TF-IDF and E5

Train three score classifiers: L2-regularized logistic classification, kNN,
and XGBoost. Compare **TF-IDF** and frozen **E5 with four extra features
(388 dimensions)**. Each question has a score distribution for each of the
11 candidate LLMs over the fixed values **0, 0.25, 0.5, 0.75 and 1**.

Read each distribution in two ways:

- **Hard class:** the score with the highest predicted probability.
- **Expected score:** the probability-weighted average of the five scores.

Both outputs share the same predicted dollar cost. The experiment uses train,
calibration and validation. Training-only cross-validation selects the logistic
regularization strength. Calibration predictions are saved for later work;
no calibrator is fitted here. Validation evaluates the resulting predictions.


## 1. Environment and paths

Use a Python environment with NumPy, pandas, SciPy, scikit-learn >= 1.6,
XGBoost >= 2.0, joblib, IPython and Jinja2. The prepared preprocessing objects
should be used with compatible package versions.

Keep the prepared files in this layout:

```text
project/
  data/processed/
    char_tfidf_50k/
    multilingual_e5_small_chunk480/
  2_Router_Training/
    TF-IDF_E5_Score_Classification_Router.ipynb
```

A layout with `1_data_preparation/data/processed/` is also supported. The code
searches the current working directory and its parents. If the data lives
elsewhere, set `DATA_PREPARATION_DIR` to the folder containing `data/`, or set
the `ERDOS_DATA_DIR` environment variable. No user-specific paths are required.
Copy the prepared data with the notebook, or run the project's data-preparation
steps before using this file.

`RUN_MODE="full"` uses all prepared rows. `"smoke"` runs a small execution check;
its metrics are not a performance estimate. Each run gets a new output folder.


In [1]:
from pathlib import Path
from datetime import datetime, timezone
from time import perf_counter
import inspect
import json
import os
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn
import xgboost
from IPython.display import display
from scipy.sparse import load_npz
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.model_selection import GroupKFold
from sklearn.neighbors import NearestNeighbors
from xgboost import XGBClassifier, XGBRegressor


In [2]:
RUN_MODE = "full"
DATA_PREPARATION_DIR = None  # Optional: folder containing data/.
OUTPUT_BASE_OVERRIDE = None  # Optional: output folder outside the project.
SEED = 42
SPLITS = ("train", "calibration", "validation")
SMOKE_LIMITS = {"train": 512, "calibration": 128, "validation": 256}

score_support = np.array([0.0, 0.25, 0.5, 0.75, 1.0])
K_VALUES = [1, 2, 3, 5, 10, 20, 40]
DISPLAY_K = 40
QUERY_BATCH_SIZE = 128

LINEAR_C_GRID = np.array([0.1, 1.0, 10.0])
LINEAR_MAX_ITER = 300
LINEAR_RETRY_MAX_ITER = (1000, 2000)
LINEAR_TOL = 1e-3
COST_RIDGE_ALPHA = {"tfidf": 10.0, "e5_388": 1.0}

XGB_DEVICE = "cpu"  # Set to "cuda" with a supported GPU installation.
XGB_N_JOBS = 4
XGB_ROUNDS = 10 if RUN_MODE == "smoke" else 100

if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'.")
if XGB_DEVICE not in {"cpu", "cuda"}:
    raise ValueError("XGB_DEVICE must be 'cpu' or 'cuda'.")
if DISPLAY_K not in K_VALUES:
    raise ValueError("DISPLAY_K must be in K_VALUES.")


def resolve_preparation_dir(override=None):
    """Find a prepared-data folder, preferring the closest project root."""
    requested = override or os.environ.get("ERDOS_DATA_DIR")
    if requested:
        candidates = [Path(requested).expanduser().resolve()]
    else:
        working_dir = Path.cwd().resolve()
        candidates = [
            candidate
            for parent in (working_dir, *working_dir.parents)
            for candidate in (parent, parent / "1_data_preparation")
        ]
    required = (
        Path("data/processed/char_tfidf_50k/preprocessing.joblib"),
        Path("data/processed/multilingual_e5_small_chunk480/embedding_metadata.json"),
    )
    for candidate in dict.fromkeys(candidates):
        if all((candidate / relative).is_file() for relative in required):
            return candidate.resolve()
    raise FileNotFoundError(
        "Prepared TF-IDF and E5 files were not found. Set DATA_PREPARATION_DIR "
        "or ERDOS_DATA_DIR to the folder containing data/processed/."
    )


preparation_dir = resolve_preparation_dir(DATA_PREPARATION_DIR)
table_dir = preparation_dir / "data/processed/char_tfidf_50k"
e5_dir = preparation_dir / "data/processed/multilingual_e5_small_chunk480"
project_dir = preparation_dir.parent if preparation_dir.name == "1_data_preparation" else preparation_dir
output_base = (
    Path(OUTPUT_BASE_OVERRIDE).expanduser().resolve()
    if OUTPUT_BASE_OVERRIDE is not None
    else project_dir / "2_Router_Training/experiment_outputs/score_classification_5class"
)
run_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
run_dir = output_base / f"{RUN_MODE}_seed{SEED}_{run_stamp}"
run_dir.mkdir(parents=True, exist_ok=False)
print("Mode:", RUN_MODE)
print("Prepared data:", preparation_dir)
print("Outputs:", run_dir)


Mode: full
Prepared data: C:\Users\johnh\MachineLearning\project_erdos
Outputs: C:\Users\johnh\MachineLearning\project_erdos\2_Router_Training\experiment_outputs\score_classification_5class\full_seed42_20261013T080041_062763Z


## 2. Load the prepared inputs

Reuse the question, score and dollar-cost tables, pure TF-IDF features, and
388-dimensional E5 features. Row indices and LLM column order must agree.
The original score tables stay unchanged throughout this notebook.


In [3]:
preprocessing = joblib.load(table_dir / "preprocessing.joblib")
tfidf_vectorizer_50k = preprocessing["tfidf_vectorizer"]
model_names = list(preprocessing["model_names"])
assert len(model_names) == 11
n_tfidf_features = len(tfidf_vectorizer_50k.vocabulary_)
e5_length_scaler = joblib.load(e5_dir / "length_scaler.joblib")
e5_count_scaler = joblib.load(e5_dir / "count_scaler.joblib")
feature_labels = {"tfidf": "TF-IDF", "e5_388": "E5 + features (388)"}
full_tables, full_features = {}, {feature: {} for feature in feature_labels}
for split in SPLITS:
    full_tables[split] = {name: pd.read_pickle(table_dir / f"{split}_{name}.pkl")
                         for name in ("questions", "scores", "costs")}
    q, s, c = (full_tables[split][name] for name in ("questions", "scores", "costs"))
    assert q.index.is_unique and q.index.equals(s.index) and q.index.equals(c.index)
    assert s.columns.tolist() == c.columns.tolist() == model_names
    assert np.isfinite(s.to_numpy(dtype=float)).all() and s.ge(0).all().all() and s.le(1).all().all()
    assert np.isfinite(c.to_numpy(dtype=float)).all() and c.gt(0).all().all()
    full_features["tfidf"][split] = load_npz(table_dir / f"X_{split}.npz")[:, :n_tfidf_features].astype(np.float32).tocsr()
    full_features["e5_388"][split] = np.load(e5_dir / f"X_{split}_e5_full_features.npy", allow_pickle=False)
    assert full_features["tfidf"][split].shape == (len(q), n_tfidf_features)
    assert full_features["e5_388"][split].shape == (len(q), 388)

# Keep the full tables for label coverage; only the execution subset changes.
tables, split_features = {}, {feature: {} for feature in feature_labels}
for split in SPLITS:
    q = full_tables[split]["questions"]
    positions = np.arange(len(q))
    if RUN_MODE == "smoke":
        rng = np.random.default_rng(SEED + SPLITS.index(split))
        positions = rng.choice(len(q), min(SMOKE_LIMITS[split], len(q)), replace=False)
        if split == "train":
            # Include each observed score to exercise the fixed mapping in smoke mode.
            scores = full_tables[split]["scores"].to_numpy(dtype=float)
            examples = [np.argwhere(scores == value)[0, 0] for value in np.unique(scores)]
            positions = np.unique(np.r_[positions, examples])
        positions.sort()
    tables[split] = {name: frame.iloc[positions].copy() for name, frame in full_tables[split].items()}
    for feature in feature_labels:
        split_features[feature][split] = full_features[feature][split][positions].copy()
    tables[split]["questions"][["sample_id"]].to_csv(run_dir / f"{split}_rows.csv")
train_questions, train_scores, train_costs = (tables["train"][name] for name in ("questions", "scores", "costs"))
validation_scores, validation_costs = (tables["validation"][name] for name in ("scores", "costs"))
train_scores_arr = train_scores.to_numpy(dtype=float)  # Original recorded scores.
cost_means = train_costs.to_numpy(dtype=float).mean(axis=0)
train_cost_normalized = (train_costs.to_numpy(dtype=float) / cost_means).astype(np.float32)
train_groups = train_questions["question_group"].to_numpy()
assert len(train_questions) >= max(K_VALUES)
all_questions = pd.concat([tables[split]["questions"].assign(_split=split) for split in SPLITS])
assert all_questions["sample_id"].is_unique
for group_key in ("question_group", "question_text"):
    assert all_questions.groupby(group_key)["_split"].nunique().eq(1).all()
display(pd.DataFrame({"Questions": {split: len(tables[split]["questions"]) for split in SPLITS}}))
print("Fixed score classes:", score_support.tolist())


C:\Users\johnh\Documents\Codex\2026-09-25\https-arxiv-org-abs-2406-18665\work\knn_verify_deps\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\johnh\Documents\Codex\2026-09-25\https-arxiv-org-abs-2406-18665\work\knn_verify_deps\sklearn\base.py:442: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.9.0 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\johnh\Documents\Codex\2026-09-25\https-arxiv-org-abs-2406-18665\work\knn_verify_deps\sklearn\b

,Questions
train,21900
calibration,3648
validation,3649


Fixed score classes: [0.0, 0.25, 0.5, 0.75, 1.0]


## 3. Why five score classes?

The five anchors cover **99.858% of training scores**. The table below counts
individual question-LLM scores in the prepared files, before any mapping or
smoke subsetting. The other values are sparse and provide little support for
learning separate classes.

| Recorded score | Train count | Calibration count | Validation count |
|---|---:|---:|---:|
| 0 | 93,268 | 15,532 | 15,346 |
| 0.25 | 15,064 | 2,417 | 2,418 |
| 0.5 | 8,075 | 1,304 | 1,382 |
| 0.75 | 27,679 | 4,599 | 4,561 |
| 1 | 96,473 | 16,203 | 16,386 |
| Other values | 341 | 73 | 46 |
| **Five-anchor coverage** | **99.858%** | **99.818%** | **99.885%** |

Using five classes gives a compact common output schema and reduces the number
of sparsely supported categories. It is a practical simplification; it does not
guarantee higher predictive accuracy or resolve uncertainty in the grading.
The next cell recomputes this table from the files loaded on your computer.

For classification, map each recorded score $s$ to its nearest fixed anchor:

$$z(s)=\arg\min_{j\in\{0,\ldots,4\}}|s-a_j|,
\qquad a=(0,\,0.25,\,0.5,\,0.75,\,1).$$

An exact midpoint uses the lower anchor. Current non-anchor values map as follows:

| Recorded values | Classification target |
|---|---:|
| 0.1 | 0 |
| 0.2, 0.3 | 0.25 |
| 0.4, 0.6 | 0.5 |
| 0.7, 0.8 | 0.75 |
| 0.9 | 1 |

The anchors and rule are fixed before evaluation. All questions remain in the
experiment. Classification accuracy uses mapped targets. MSE, MAE and WAPE use
the **original recorded scores**, so their meaning is preserved.


In [4]:
coverage_rows = []
for label in [*score_support, "Other values"]:
    row = {"Recorded score": label}
    for split in SPLITS:
        values = full_tables[split]["scores"].to_numpy(dtype=float)
        row[split] = int(
            (~np.isin(values, score_support)).sum()
            if isinstance(label, str) else (values == label).sum()
        )
    coverage_rows.append(row)
label_coverage_counts = pd.DataFrame(coverage_rows).set_index("Recorded score")
anchor_coverage = pd.DataFrame({
    split: {
        "Total score entries": full_tables[split]["scores"].size,
        "Five-anchor coverage": np.isin(
            full_tables[split]["scores"].to_numpy(dtype=float), score_support,
        ).mean(),
    }
    for split in SPLITS
}).T
display(label_coverage_counts)
display(anchor_coverage.style.format({
    "Total score entries": "{:,.0f}", "Five-anchor coverage": "{:.3%}",
}))
label_coverage_counts.to_csv(run_dir / "raw_score_value_counts.csv")
anchor_coverage.to_csv(run_dir / "raw_score_anchor_coverage.csv")


,train,calibration,validation
Recorded score,,,
0.0,93268,15532,15346
0.25,15064,2417,2418
0.5,8075,1304,1382
0.75,27679,4599,4561
1.0,96473,16203,16386
Other values,341,73,46


,Total score entries,Five-anchor coverage
train,"240,900",99.858%
calibration,"40,128",99.818%
validation,"40,139",99.885%


In [5]:
def encode_score_classes(values):
    """Map original scores to fixed anchors without changing the source array."""
    values = np.asarray(values, dtype=float)
    if not np.isfinite(values).all() or np.any((values < 0) | (values > 1)):
        raise ValueError("Scores must be finite and lie in [0, 1].")
    distances = np.abs(values[..., None] - score_support)
    return distances.argmin(axis=-1).astype(np.int64)


def align_class_probabilities(probabilities, classes, n_classes):
    probabilities = np.asarray(probabilities, dtype=float)
    classes = np.asarray(classes, dtype=int)
    aligned = np.zeros((len(probabilities), n_classes), dtype=float)
    aligned[:, classes] = probabilities
    if not np.isfinite(aligned).all() or np.any(aligned < 0) or not np.allclose(aligned.sum(axis=1), 1, atol=1e-6):
        raise ValueError("Invalid score probability distribution.")
    return aligned


def class_log_loss(targets, probabilities, floor=1e-12):
    true_probability = probabilities[np.arange(len(targets)), targets]
    return float(-np.log(np.maximum(true_probability, floor)).mean())


def prediction_frame(values, split):
    return pd.DataFrame(values, index=tables[split]["questions"].index, columns=model_names)


score_class_ids = encode_score_classes(train_scores_arr)
training_folds = list(GroupKFold(n_splits=3, shuffle=True, random_state=SEED).split(
    train_scores_arr, groups=train_groups,
))
for fit_indices, heldout_indices in training_folds:
    assert not np.intersect1d(train_groups[fit_indices], train_groups[heldout_indices]).size
e5_train = split_features["e5_388"]["train"]
raw_lengths = e5_length_scaler.inverse_transform(e5_train[:, 384:385])
raw_counts = e5_count_scaler.inverse_transform(e5_train[:, 385:387])
training_texts = train_questions["question_text"].to_numpy()


def build_fold_features(feature, fit_indices, heldout_indices):
    if feature == "tfidf":
        vectorizer = clone(tfidf_vectorizer_50k)
        return (
            vectorizer.fit_transform(training_texts[fit_indices]).astype(np.float32).tocsr(),
            vectorizer.transform(training_texts[heldout_indices]).astype(np.float32).tocsr(),
        )
    length_scaler = clone(e5_length_scaler).fit(raw_lengths[fit_indices])
    count_scaler = clone(e5_count_scaler).fit(raw_counts[fit_indices])

    def transform(positions):
        return np.column_stack([e5_train[positions, :384],
                                length_scaler.transform(raw_lengths[positions]),
                                count_scaler.transform(raw_counts[positions]),
                                e5_train[positions, 387:388]]).astype(np.float32)

    return transform(fit_indices), transform(heldout_indices)


prediction_bundles = {}


## 4. L2-regularized logistic classification

Train one classifier per candidate LLM. Each classifier produces probabilities
for the five anchors. Cross-entropy rewards probability assigned to the correct
mapped class, and L2 regularization controls coefficient size:

$$\mathcal L_{\mathrm{score}}=
-\frac1n\sum_{i=1}^n\log p_{i,z_i}
+\frac\lambda2\|W\|_F^2.$$

In scikit-learn, a smaller `C` means stronger regularization. Select one `C` per
feature set by mean cross-entropy over three training folds and the 11 LLMs.
Question groups stay within one fold. TF-IDF and the extra-feature scalers are
refitted inside each fold; the frozen E5 vectors are reused.

The price head is a multi-output Ridge model. For LLM $m$, let
$\mu_m=\operatorname{mean}_{\mathrm{train}}c_{im}$. Fit normalized costs
$c_{im}/\mu_m$ with a squared-error objective and L2 penalty. Clip negative
estimates to a small positive value and multiply by $\mu_m$ to restore dollars.


In [6]:
L2_OPTIONS = {"penalty": "l2"} if inspect.signature(LogisticRegression).parameters["l1_ratio"].default is None else {"l1_ratio": 0.0}


def fit_linear_score_head(features, targets, C):
    """Return a classifier, or its constant class, with fit diagnostics."""
    classes = np.unique(targets)
    if classes.size == 1:
        return {
            "classifier": None, "constant_class": int(classes[0]),
            "iterations": 0, "converged": True, "observed_classes": classes,
            "convergence_warning_count": 0,
        }
    classifier = LogisticRegression(
        C=float(C), solver="lbfgs", max_iter=LINEAR_MAX_ITER, **L2_OPTIONS,
        tol=LINEAR_TOL, random_state=SEED,
    )
    total_iterations, warning_count = 0, 0
    for iteration_limit in (LINEAR_MAX_ITER, *LINEAR_RETRY_MAX_ITER):
        classifier.set_params(max_iter=iteration_limit)
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always", ConvergenceWarning)
            classifier.fit(features, targets)
        convergence_messages = [str(item.message) for item in caught
                                if issubclass(item.category, ConvergenceWarning)]
        warning_count += len(convergence_messages)
        total_iterations += int(classifier.n_iter_.max())
        for item in caught:
            if not issubclass(item.category, ConvergenceWarning):
                warnings.warn(str(item.message), item.category)
        if not convergence_messages:
            break
        # Continue from the current coefficients with the same objective and tolerance.
        classifier.set_params(warm_start=True)
        print(f"    iteration limit {iteration_limit}: continuing optimization", flush=True)
    if not np.isfinite(classifier.coef_).all() or not np.isfinite(classifier.intercept_).all():
        raise ValueError("A linear score classifier produced nonfinite parameters.")
    return {
        "classifier": classifier, "constant_class": None,
        "iterations": total_iterations,
        "converged": not convergence_messages,
        "observed_classes": classifier.classes_.astype(int),
        "convergence_warning_count": warning_count,
    }


def linear_score_probabilities(head, features):
    if head["classifier"] is None:
        probabilities = np.zeros((features.shape[0], len(score_support)))
        probabilities[:, head["constant_class"]] = 1.0
        return probabilities
    classifier = head["classifier"]
    return align_class_probabilities(
        classifier.predict_proba(features), classifier.classes_,
        len(score_support),
    )


In [7]:
linear_cv_rows = []
for feature in feature_labels:
    for fold_id, (fit_indices, heldout_indices) in enumerate(training_folds, 1):
        print(f"Linear score CV / {feature_labels[feature]} / fold {fold_id}: features", flush=True)
        X_fit, X_heldout = build_fold_features(feature, fit_indices, heldout_indices)
        for C in LINEAR_C_GRID:
            started = perf_counter()
            losses = []
            unseen_labels = 0
            n_unconverged = 0
            warning_count = 0
            max_iterations = 0
            max_observed_classes = 0
            for model_position, model_name in enumerate(model_names):
                head = fit_linear_score_head(
                    X_fit, score_class_ids[fit_indices, model_position], C,
                )
                probabilities = linear_score_probabilities(head, X_heldout)
                targets = score_class_ids[heldout_indices, model_position]
                losses.append(class_log_loss(targets, probabilities))
                unseen_labels += int((~np.isin(targets, head["observed_classes"])).sum())
                n_unconverged += int(not head["converged"])
                warning_count += head["convergence_warning_count"]
                max_iterations = max(max_iterations, head["iterations"])
                max_observed_classes = max(max_observed_classes, len(head["observed_classes"]))
                print(
                    f"  C={C:g} / {model_position + 1}/{len(model_names)} / "
                    f"{model_name}: CE={losses[-1]:.4f}, "
                    f"iterations={head['iterations']}, converged={head['converged']}",
                    flush=True,
                )
            linear_cv_rows.append({
                "Feature": feature, "Fold": fold_id, "C": float(C),
                "Questions": len(heldout_indices), "Mean_CrossEntropy": np.mean(losses),
                "Unseen_label_events": unseen_labels,
                "Unconverged_heads": n_unconverged, "Max_iterations": max_iterations,
                "Convergence_warnings": warning_count, "Max_observed_classes": max_observed_classes,
                "Seconds": perf_counter() - started,
            })
        del X_fit, X_heldout

linear_cv_table = pd.DataFrame(linear_cv_rows)
linear_cv_summary_rows = []
for (feature, C), rows in linear_cv_table.groupby(["Feature", "C"], sort=False):
    linear_cv_summary_rows.append({
        "Feature": feature, "C": C,
        "CV_CrossEntropy": np.average(rows["Mean_CrossEntropy"], weights=rows["Questions"]),
        "Unseen_label_events": int(rows["Unseen_label_events"].sum()),
        "Unconverged_heads": int(rows["Unconverged_heads"].sum()),
        "Convergence_warnings": int(rows["Convergence_warnings"].sum()),
        "Max_iterations": int(rows["Max_iterations"].max()),
        "Max_observed_classes": int(rows["Max_observed_classes"].max()),
        "Seconds": rows["Seconds"].sum(),
    })
linear_cv_summary = pd.DataFrame(linear_cv_summary_rows)
selected_linear_C = {}
for feature in feature_labels:
    choices = linear_cv_summary.loc[linear_cv_summary["Feature"] == feature]
    # Exact ties favor the smaller C, which gives stronger regularization.
    best = choices.sort_values(["CV_CrossEntropy", "C"], kind="stable").iloc[0]
    selected_linear_C[feature] = float(best["C"])
display(linear_cv_summary)
print("Selected score-classifier C:", selected_linear_C)
if linear_cv_summary["Unconverged_heads"].sum():
    warnings.warn("Some CV fits reached the iteration limit; increase LINEAR_MAX_ITER before interpreting close results.")


Linear score CV / TF-IDF / fold 1: features
Parallel linear: 33 tasks, 3 processes
Completed 1/33: {'model_position': 2, 'seconds': 8.72942000022158, 'fits': [{'kind': 'linear', 'C': 0.1, 'converged': True, 'iterations': 25, 'path': 'C:\\Users\\johnh\\MachineLearning\\project_erdos\\2_Router_Training\\experiment_outputs\\score_classification_5class\\full_seed42_20261013T080041_062763Z\\fit_checkpoints\\head_linear_tfidf_fold1_m2_C0.1.joblib'}]}
Completed 2/33: {'model_position': 1, 'seconds': 9.478393900208175, 'fits': [{'kind': 'linear', 'C': 0.1, 'converged': True, 'iterations': 27, 'path': 'C:\\Users\\johnh\\MachineLearning\\project_erdos\\2_Router_Training\\experiment_outputs\\score_classification_5class\\full_seed42_20261013T080041_062763Z\\fit_checkpoints\\head_linear_tfidf_fold1_m1_C0.1.joblib'}]}
Completed 3/33: {'model_position': 0, 'seconds': 9.714522399939597, 'fits': [{'kind': 'linear', 'C': 0.1, 'converged': True, 'iterations': 24, 'path': 'C:\\Users\\johnh\\MachineLearnin

,Feature,C,CV_CrossEntropy,Unseen_label_events,Unconverged_heads,Convergence_warnings,Max_iterations,Max_observed_classes,Seconds
0,tfidf,0.1,0.672700,0,0,0,32,5,604.002780
1,tfidf,1.0,0.644240,0,0,0,62,5,2.178599
2,tfidf,10.0,0.741606,0,0,0,121,5,2.161826
3,e5_388,0.1,0.666306,0,0,0,49,5,104.429894
4,e5_388,1.0,0.641514,0,0,0,90,5,0.623219
5,e5_388,10.0,0.642943,0,0,0,152,5,0.640777


Selected score-classifier C: {'tfidf': 1.0, 'e5_388': 1.0}


In [8]:
linear_models = {}
linear_fit_rows = []
for feature in feature_labels:
    C = selected_linear_C[feature]
    X_train = split_features[feature]["train"]
    heads = []
    for model_position, model_name in enumerate(model_names):
        started = perf_counter()
        head = fit_linear_score_head(X_train, score_class_ids[:, model_position], C)
        heads.append(head)
        linear_fit_rows.append({
            "Feature": feature, "Model": model_name, "C": C,
            "Iterations": head["iterations"], "Converged": head["converged"],
            "Convergence_warnings": head["convergence_warning_count"],
            "Observed_classes": len(head["observed_classes"]),
            "Seconds": perf_counter() - started,
        })
        print(f"Logistic / {feature_labels[feature]} / {model_position + 1}/{len(model_names)}: {model_name}")

    cost_model = Ridge(
        alpha=COST_RIDGE_ALPHA[feature], solver="lsqr", tol=1e-3, max_iter=1000,
    ).fit(X_train, train_cost_normalized)
    linear_models[feature] = {"heads": heads, "cost_model": cost_model, "selected_C": C}
    for split in ("calibration", "validation"):
        X = split_features[feature][split]
        probabilities = np.stack([linear_score_probabilities(head, X) for head in heads], axis=1)
        costs = np.maximum(cost_model.predict(X), 1e-6) * cost_means[None, :]
        prediction_bundles[("L2 logistic", feature, split)] = {
            "probability": probabilities, "cost": prediction_frame(costs, split),
        }

linear_fit_table = pd.DataFrame(linear_fit_rows)
display(linear_fit_table)
if (~linear_fit_table["Converged"]).any():
    warnings.warn("Some final classifiers did not converge; check the fit diagnostics.")


Parallel linear: 11 tasks, 3 processes
Completed 1/11: {'model_position': 2, 'seconds': 19.158276699949056, 'fits': [{'kind': 'linear', 'C': 1.0, 'converged': True, 'iterations': 38, 'path': 'C:\\Users\\johnh\\MachineLearning\\project_erdos\\2_Router_Training\\experiment_outputs\\score_classification_5class\\full_seed42_20261013T080041_062763Z\\fit_checkpoints\\head_linear_tfidf_fold0_m2_C1.joblib'}]}
Completed 2/11: {'model_position': 0, 'seconds': 21.21010740008205, 'fits': [{'kind': 'linear', 'C': 1.0, 'converged': True, 'iterations': 41, 'path': 'C:\\Users\\johnh\\MachineLearning\\project_erdos\\2_Router_Training\\experiment_outputs\\score_classification_5class\\full_seed42_20261013T080041_062763Z\\fit_checkpoints\\head_linear_tfidf_fold0_m0_C1.joblib'}]}
Completed 3/11: {'model_position': 1, 'seconds': 30.42770090000704, 'fits': [{'kind': 'linear', 'C': 1.0, 'converged': True, 'iterations': 64, 'path': 'C:\\Users\\johnh\\MachineLearning\\project_erdos\\2_Router_Training\\experimen

,Feature,Model,C,Iterations,Converged,Convergence_warnings,Observed_classes,Seconds
0,tfidf,WizardLM/WizardLM-13B-V1.2,1.0,41,True,0,5,91.418612
1,tfidf,claude-instant-v1,1.0,64,True,0,5,0.002589
2,tfidf,claude-v1,1.0,38,True,0,5,0.003512
3,tfidf,claude-v2,1.0,42,True,0,5,0.003430
4,tfidf,gpt-3.5-turbo-1106,1.0,39,True,0,5,0.002869
5,tfidf,gpt-4-1106-preview,1.0,36,True,0,5,0.003094
6,tfidf,meta/code-llama-instruct-34b-chat,1.0,49,True,0,5,0.002685
7,tfidf,meta/llama-2-70b-chat,1.0,64,True,0,5,0.002384
8,tfidf,mistralai/mistral-7b-chat,1.0,41,True,0,5,0.001929
9,tfidf,mistralai/mixtral-8x7b-chat,1.0,43,True,0,5,0.002096


Read the cross-validation and convergence tables before interpreting validation
differences. A selected `C` at either end of the grid suggests exploring a wider
grid in a later experiment. The price penalty is fixed separately for each
feature set. Only the score classifier's `C` is tuned in this notebook.


## 5. kNN class frequencies and mean dollar cost

Retrieve training neighbors with cosine distance:

$$d(x,x')=1-\frac{x^\mathsf T x'}{\|x\|\|x'\|}.$$

For $K$ neighbors, count their **mapped** score classes for each candidate LLM:

$$p_{mj}(q)=\frac1K\sum_{i\in N_K(q)}\mathbf1[z(s_{im})=j],
\qquad
\hat c_m(q)=\frac1K\sum_{i\in N_K(q)}c_{im}.$$

Expected score is the average of the mapped neighbor scores. Hard class is the
most frequent mapped score. Price remains the mean of the recorded neighbor
costs. Query in batches, reuse the same neighbors for every requested $K$, and
use `DISPLAY_K=40` in the main comparison. Keep all $K$ results in a separate table.


In [9]:
def predict_neighbor_classes(index, features, split):
    count = features.shape[0]
    n_classes = len(score_support)
    predictions = {
        k: {"probability": np.empty((count, len(model_names), n_classes)),
            "cost": np.empty((count, len(model_names)))}
        for k in K_VALUES
    }
    train_cost_values = train_costs.to_numpy(dtype=float)
    started = perf_counter()
    for start in range(0, count, QUERY_BATCH_SIZE):
        stop = min(start + QUERY_BATCH_SIZE, count)
        neighbors = index.kneighbors(
            features[start:stop], n_neighbors=max(K_VALUES), return_distance=False,
        )
        labels = score_class_ids[neighbors]
        cost_sums = np.cumsum(train_cost_values[neighbors], axis=1)
        class_counts = np.zeros((stop - start, len(model_names), n_classes))
        rows, models = np.indices((stop - start, len(model_names)))
        for position in range(max(K_VALUES)):
            class_counts[rows, models, labels[:, position, :]] += 1
            k = position + 1
            if k in predictions:
                predictions[k]["probability"][start:stop] = class_counts / k
                predictions[k]["cost"][start:stop] = cost_sums[:, position, :] / k
        print(f"kNN / {split}: {stop}/{count} questions | {perf_counter() - started:.1f}s", flush=True)
    for bundle in predictions.values():
        bundle["cost"] = prediction_frame(bundle["cost"], split)
    return predictions


In [10]:
knn_indices, knn_class_predictions = {}, {}
for feature in feature_labels:
    index = NearestNeighbors(metric="cosine", algorithm="brute", n_jobs=-1).fit(
        split_features[feature]["train"],
    )
    knn_indices[feature] = index
    knn_class_predictions[feature] = {}
    for split in ("calibration", "validation"):
        all_k = predict_neighbor_classes(index, split_features[feature][split], split)
        knn_class_predictions[feature][split] = all_k
        prediction_bundles[("kNN", feature, split)] = all_k[DISPLAY_K]


kNN / calibration: 128/3648 questions | 0.7s
kNN / calibration: 256/3648 questions | 1.5s
kNN / calibration: 384/3648 questions | 2.7s
kNN / calibration: 512/3648 questions | 4.2s
kNN / calibration: 640/3648 questions | 5.7s
kNN / calibration: 768/3648 questions | 7.2s
kNN / calibration: 896/3648 questions | 8.7s
kNN / calibration: 1024/3648 questions | 10.2s
kNN / calibration: 1152/3648 questions | 11.3s
kNN / calibration: 1280/3648 questions | 12.3s
kNN / calibration: 1408/3648 questions | 13.6s
kNN / calibration: 1536/3648 questions | 14.9s
kNN / calibration: 1664/3648 questions | 16.2s
kNN / calibration: 1792/3648 questions | 17.5s
kNN / calibration: 1920/3648 questions | 18.8s
kNN / calibration: 2048/3648 questions | 20.0s
kNN / calibration: 2176/3648 questions | 21.0s
kNN / calibration: 2304/3648 questions | 22.1s
kNN / calibration: 2432/3648 questions | 23.1s
kNN / calibration: 2560/3648 questions | 24.1s
kNN / calibration: 2688/3648 questions | 25.2s
kNN / calibration: 2816/364

## 6. XGBoost score classification and price prediction

Fit one boosted-tree score classifier and one price model per LLM. The score
head uses multiclass cross-entropy (`multi:softprob`). If a training subset has
only two observed classes, it uses binary cross-entropy. A constant target
produces a constant distribution. In every case, align the output to the same
five class columns and fill unobserved classes with zero probability.

The price head minimizes squared error on normalized costs. Restore dollar
units with the training cost means. Both hard and expected score outputs share
these price predictions. Tree settings are fixed in this experiment; XGBoost
hyperparameters are not selected by cross-validation here.


In [11]:
xgb_params = {
    "n_estimators": XGB_ROUNDS, "max_depth": 3, "learning_rate": 0.05,
    "subsample": 0.8, "colsample_bytree": 0.8, "max_bin": 64,
    "tree_method": "hist", "device": XGB_DEVICE, "n_jobs": XGB_N_JOBS,
    "reg_lambda": 1.0, "random_state": SEED,
}


def fit_xgb_score_head(features, targets):
    classes, encoded = np.unique(targets, return_inverse=True)
    if len(classes) == 1:
        return {"classifier": None, "classes": classes}
    objective = "binary:logistic" if len(classes) == 2 else "multi:softprob"
    options = {"eval_metric": "logloss" if len(classes) == 2 else "mlogloss"}
    if len(classes) > 2:
        options["num_class"] = len(classes)
    classifier = XGBClassifier(objective=objective, **options, **xgb_params).fit(features, encoded)
    actual_device = json.loads(classifier.get_booster().save_config())["learner"]["generic_param"]["device"]
    if XGB_DEVICE == "cuda" and not actual_device.startswith("cuda"):
        raise RuntimeError("XGBoost fell back to CPU. Set XGB_DEVICE='cpu' or check the GPU installation.")
    return {"classifier": classifier, "classes": classes}


def xgb_score_probabilities(head, features):
    if head["classifier"] is None:
        probabilities = np.zeros((features.shape[0], len(score_support)))
        probabilities[:, head["classes"][0]] = 1.0
        return probabilities
    return align_class_probabilities(head["classifier"].predict_proba(features), head["classes"], len(score_support))


In [12]:
xgb_models = {}
for feature in feature_labels:
    X_train = split_features[feature]["train"]
    xgb_models[feature] = {"score_classifier": [], "cost_regressor": []}
    raw = {split: {"probability": [], "cost": []} for split in ("calibration", "validation")}
    for model_position, model_name in enumerate(model_names):
        started = perf_counter()
        print(f"XGBoost / {feature_labels[feature]} / {model_position + 1}/{len(model_names)}: {model_name}", flush=True)
        head = fit_xgb_score_head(X_train, score_class_ids[:, model_position])
        price_head = XGBRegressor(objective="reg:squarederror", **xgb_params).fit(
            X_train, train_cost_normalized[:, model_position],
        )
        xgb_models[feature]["score_classifier"].append(head)
        xgb_models[feature]["cost_regressor"].append(price_head)
        for split in raw:
            X = split_features[feature][split]
            raw[split]["probability"].append(xgb_score_probabilities(head, X))
            costs = np.maximum(price_head.predict(X), 1e-6) * cost_means[model_position]
            raw[split]["cost"].append(costs)
        print(f"  completed in {perf_counter() - started:.1f}s", flush=True)
    for split, values in raw.items():
        prediction_bundles[("XGBoost", feature, split)] = {
            "probability": np.stack(values["probability"], axis=1),
            "cost": prediction_frame(np.column_stack(values["cost"]), split),
        }


XGBoost / TF-IDF / 1/11: WizardLM/WizardLM-13B-V1.2
Parallel xgb: 11 tasks, 2 processes
Completed 1/11: {'model_position': 1, 'seconds': 279.73125259997323, 'fits': [{'kind': 'score', 'cached': False, 'path': 'C:\\Users\\johnh\\MachineLearning\\project_erdos\\2_Router_Training\\experiment_outputs\\score_classification_5class\\full_seed42_20261013T080041_062763Z\\fit_checkpoints\\head_xgb_score_tfidf_fold0_m1.joblib', 'seconds': 225.06367429997772}, {'kind': 'cost', 'cached': False, 'path': 'C:\\Users\\johnh\\MachineLearning\\project_erdos\\2_Router_Training\\experiment_outputs\\score_classification_5class\\full_seed42_20261013T080041_062763Z\\fit_checkpoints\\head_xgb_cost_tfidf_fold0_m1.joblib', 'seconds': 54.62288610031828}]}
Completed 2/11: {'model_position': 0, 'seconds': 285.1203362001106, 'fits': [{'kind': 'score', 'cached': False, 'path': 'C:\\Users\\johnh\\MachineLearning\\project_erdos\\2_Router_Training\\experiment_outputs\\score_classification_5class\\full_seed42_20261013T08

## 7. Evaluate probabilities and the two score outputs

For question $i$ and candidate LLM $m$:

$$\hat s^{\mathrm{hard}}_{im}=a_{\arg\max_j p_{imj}},
\qquad \hat s^{\mathrm{expected}}_{im}=\sum_{j=0}^4 a_jp_{imj}.$$

Hard class belongs to the five anchors. Expected score can take an intermediate
value, such as 0.63. An exact probability tie uses the lower anchor because the
class columns are ordered from 0 to 1.

**Probability metrics** evaluate the five-class target $z_{im}$:

$$\mathrm{Accuracy}_m=\frac1n\sum_i
\mathbf1[\arg\max_j p_{imj}=z_{im}],
\qquad
\mathrm{NLL}_m=-\frac1n\sum_i\log\max(p_{im,z_{im}},10^{-12}).$$

Balanced accuracy averages recall across the classes present in validation.
It reveals performance hidden by frequent classes. Report the fraction of
zero true-class probabilities alongside NLL; the numerical floor keeps NLL
finite. Accuracy measures hard class decisions, while NLL evaluates the whole
distribution.

**Numerical error metrics** compare both score outputs with the original recorded
score $s_{im}$. Predicted costs are compared with recorded dollar costs $c_{im}$:

$$\mathrm{MSE}_m=\frac1n\sum_i(\hat y_{im}-y_{im})^2,
\qquad \mathrm{MAE}_m=\frac1n\sum_i|\hat y_{im}-y_{im}|,$$

$$\mathrm{WAPE}_m=
\frac{\sum_i|\hat y_{im}-y_{im}|}{\sum_i|y_{im}|}.$$

Compute each metric separately for each LLM, then average equally across the
11 LLMs. WAPE avoids dividing by each individual zero score; an all-zero target
column gives an undefined WAPE and is counted explicitly. Hard and expected
outputs share price predictions, so their cost errors are identical.


In [13]:
NLL_EPSILON = 1e-12
score_support = np.asarray(score_support, dtype=float)
assert np.all(np.diff(score_support) > 0), "The score support must be strictly sorted."
validation_index = validation_scores.index
assert validation_costs.index.equals(validation_index), "Validation score/cost rows must align."
validation_truth = validation_scores.loc[:, model_names].to_numpy(dtype=float)
validation_cost_truth = validation_costs.loc[:, model_names].to_numpy(dtype=float)
n_questions, n_models = validation_truth.shape
n_score_classes = len(score_support)


validation_class_ids = encode_score_classes(validation_truth)


def checked_probability_bundle(bundle):
    probabilities = np.asarray(bundle["probability"], dtype=float)
    costs = bundle["cost"]
    if probabilities.shape != (n_questions, n_models, n_score_classes):
        raise ValueError("Probabilities must have shape (questions, candidate LLMs, classes).")
    if not np.isfinite(probabilities).all() or np.any(probabilities < 0):
        raise ValueError("Class probabilities must be finite and nonnegative.")
    if not np.allclose(probabilities.sum(axis=-1), 1.0, atol=1e-6):
        raise ValueError("Each candidate LLM needs a probability distribution summing to one.")
    if not costs.index.equals(validation_index) or costs.columns.tolist() != model_names:
        raise ValueError("Cost prediction rows and model columns must preserve validation order.")
    cost_values = costs.to_numpy(dtype=float)
    if not np.isfinite(cost_values).all() or np.any(cost_values < 0):
        raise ValueError("Predicted dollar costs must be finite and nonnegative.")
    return probabilities, cost_values


def distribution_metrics(bundle):
    probabilities, _ = checked_probability_bundle(bundle)
    hard_class_ids = probabilities.argmax(axis=-1)
    true_probabilities = np.take_along_axis(
        probabilities, validation_class_ids[..., None], axis=-1,
    )[..., 0]
    balanced_accuracy = []
    for model_position in range(n_models):
        truth = validation_class_ids[:, model_position]
        prediction = hard_class_ids[:, model_position]
        recalls = [(prediction[truth == value] == value).mean() for value in np.unique(truth)]
        balanced_accuracy.append(float(np.mean(recalls)))
    return pd.DataFrame({
        "Accuracy": (hard_class_ids == validation_class_ids).mean(axis=0),
        "Balanced_accuracy": balanced_accuracy,
        "NLL": -np.log(np.maximum(true_probabilities, NLL_EPSILON)).mean(axis=0),
        "Zero_true_probability_rate": (true_probabilities == 0).mean(axis=0),
    }, index=pd.Index(model_names, name="model"))


def score_outputs(bundle):
    probabilities, costs = checked_probability_bundle(bundle)
    expected_scores = probabilities @ score_support
    return {
        "Expected score": {"score": expected_scores, "cost": costs},
        "Hard class": {"score": score_support[probabilities.argmax(axis=-1)], "cost": costs},
    }


def relative_absolute_error(predictions, targets):
    totals = np.abs(targets).sum(axis=0)
    return np.divide(
        np.abs(predictions - targets).sum(axis=0), totals,
        out=np.full(targets.shape[1], np.nan), where=totals > 0,
    )


def numeric_prediction_metrics(bundle):
    scores, costs = bundle["score"], bundle["cost"]
    return pd.DataFrame({
        "Score_MSE": np.square(scores - validation_truth).mean(axis=0),
        "Score_MAE": np.abs(scores - validation_truth).mean(axis=0),
        "Score_WAPE": relative_absolute_error(scores, validation_truth),
        "Cost_MSE": np.square(costs - validation_cost_truth).mean(axis=0),
        "Cost_MAE": np.abs(costs - validation_cost_truth).mean(axis=0),
        "Cost_WAPE": relative_absolute_error(costs, validation_cost_truth),
    }, index=pd.Index(model_names, name="model"))


def training_frequency_bundle(train_scores, train_costs):
    train_values = train_scores.loc[:, model_names].to_numpy(dtype=float)
    train_classes = encode_score_classes(train_values)
    frequencies = np.stack([
        np.bincount(train_classes[:, model], minlength=n_score_classes) / len(train_values)
        for model in range(n_models)
    ])
    mean_costs = train_costs.loc[:, model_names].mean(axis=0).to_numpy(dtype=float)
    return {
        "probability": np.broadcast_to(frequencies, (n_questions, n_models, n_score_classes)),
        "cost": pd.DataFrame(np.broadcast_to(mean_costs, (n_questions, n_models)),
                             index=validation_index, columns=model_names),
    }


In [14]:
# prediction_bundles is keyed by (method, embedding, split).
validation_probability_bundles = {
    (method, embedding): bundle
    for (method, embedding, split), bundle in prediction_bundles.items()
    if split == "validation"
}
validation_probability_bundles = {
    ("Training frequency", "Baseline"): training_frequency_bundle(train_scores, train_costs),
    **validation_probability_bundles,
}
distribution_per_model = pd.concat({
    key: distribution_metrics(bundle)
    for key, bundle in validation_probability_bundles.items()
}, names=["method", "embedding"])
distribution_summary = distribution_per_model.groupby(
    level=["method", "embedding"], sort=False,
).mean()
numeric_validation_bundles = {
    (method, embedding, output): values
    for (method, embedding), bundle in validation_probability_bundles.items()
    for output, values in score_outputs(bundle).items()
}
numeric_per_model = pd.concat({
    key: numeric_prediction_metrics(bundle)
    for key, bundle in numeric_validation_bundles.items()
}, names=["method", "embedding", "score_output"])
numeric_summary = numeric_per_model.groupby(
    level=["method", "embedding", "score_output"], sort=False,
).mean()
for target in ("Score", "Cost"):
    numeric_summary[f"Defined_{target.lower()}_WAPE_models"] = numeric_per_model[
        f"{target}_WAPE"
    ].groupby(level=["method", "embedding", "score_output"], sort=False).count()

print("Validation probability quality: one result per trained classifier")
display(distribution_summary.style.format({
    "Accuracy": "{:.1%}", "Balanced_accuracy": "{:.1%}", "NLL": "{:.4f}",
    "Zero_true_probability_rate": "{:.1%}",
}))
print("Validation score/cost errors: two score outputs from the same probabilities")
display(numeric_summary.style.format({
    "Score_MSE": "{:.5f}", "Score_MAE": "{:.5f}", "Score_WAPE": "{:.1%}",
    "Cost_MSE": "{:.3e}", "Cost_MAE": "${:.8f}", "Cost_WAPE": "{:.1%}",
    "Defined_score_WAPE_models": "{:.0f}", "Defined_cost_WAPE_models": "{:.0f}",
}, na_rep="undefined"))
distribution_summary.to_csv(run_dir / "validation_classification_summary.csv")
distribution_per_model.to_csv(run_dir / "validation_classification_per_llm.csv")
numeric_summary.to_csv(run_dir / "validation_numeric_summary.csv")
numeric_per_model.to_csv(run_dir / "validation_numeric_per_llm.csv")


Validation probability quality: one result per trained classifier


Validation score/cost errors: two score outputs from the same probabilities


### Compare neighbor counts

The table below shows every requested `k` for both feature sets. Compare hard
and expected score errors within each `k`. Classification accuracy and NLL come
from their shared distribution, and cost errors come from their shared price
estimate. Keep validation comparisons exploratory when choosing a final `k`.


In [15]:
knn_grid_rows = []
for feature in feature_labels:
    for k in K_VALUES:
        bundle = knn_class_predictions[feature]["validation"][k]
        distribution = distribution_metrics(bundle)
        for output, scores in score_outputs(bundle).items():
            metrics = numeric_prediction_metrics(scores).mean()
            knn_grid_rows.append({"embedding": feature, "k": k, "output": output,
                                  "Score_WAPE": metrics["Score_WAPE"], "Cost_WAPE": metrics["Cost_WAPE"],
                                  "Score_MSE": metrics["Score_MSE"], "Score_MAE": metrics["Score_MAE"], "NLL": distribution["NLL"].mean(),
                                  "Accuracy": distribution["Accuracy"].mean()})
knn_grid_summary = pd.DataFrame(knn_grid_rows)
knn_grid_summary.to_csv(run_dir / "knn_score_classification_grid.csv", index=False)
display(knn_grid_summary.style.format({"Score_WAPE": "{:.1%}", "Cost_WAPE": "{:.1%}",
                                      "Score_MSE": "{:.5f}", "Score_MAE": "{:.5f}", "NLL": "{:.4f}", "Accuracy": "{:.1%}"}))


,embedding,k,output,Score_WAPE,Cost_WAPE,Score_MSE,Score_MAE,NLL,Accuracy
0,tfidf,1,Expected score,67.7%,24.3%,0.29381,0.31870,10.7512,61.1%
1,tfidf,1,Hard class,67.7%,24.3%,0.29381,0.31870,10.7512,61.1%
2,tfidf,2,Expected score,67.8%,23.0%,0.22371,0.32119,6.1788,58.5%
3,tfidf,2,Hard class,67.8%,23.0%,0.31044,0.33702,6.1788,58.5%
4,tfidf,3,Expected score,68.1%,22.9%,0.19982,0.32250,4.1228,63.5%
5,tfidf,3,Hard class,64.0%,22.9%,0.28016,0.30321,4.1228,63.5%
6,tfidf,5,Expected score,67.9%,22.9%,0.17980,0.32277,2.5076,65.2%
7,tfidf,5,Hard class,60.6%,22.9%,0.26570,0.28773,2.5076,65.2%
8,tfidf,10,Expected score,68.2%,23.2%,0.16552,0.32371,1.3621,66.6%
9,tfidf,10,Hard class,58.0%,23.2%,0.25561,0.27698,1.3621,66.6%


## 8. Save and reuse the results

Each run saves probability arrays, hard scores, expected scores and dollar costs
for calibration and validation. DataFrames preserve question indices and LLM
columns. Probability arrays use `(questions, 11 LLMs, 5 classes)`; the manifest
records both column orders. All kNN neighbor counts are exported, with `k=40`
also appearing in the main comparison.

The folder also contains metrics, class-coverage tables, training-fold records,
fit diagnostics, model settings and fitted Logistic/XGBoost objects. Saved
predictions can be loaded for further evaluation without fitting models again.
Prediction on new questions additionally requires the prepared TF-IDF/E5
workflow. kNN lookup objects can be rebuilt from the prepared training features
and mapped labels.


In [16]:
def save_prediction_bundle(prefix, bundle, split):
    probabilities = np.asarray(bundle["probability"], dtype=float)
    assert probabilities.shape == (len(tables[split]["questions"]), len(model_names), 5)
    assert np.isfinite(probabilities).all() and (probabilities >= 0).all()
    assert np.allclose(probabilities.sum(axis=-1), 1, atol=1e-6)
    np.save(run_dir / f"{prefix}_probability.npy", probabilities, allow_pickle=False)
    prediction_frame(probabilities @ score_support, split).to_pickle(run_dir / f"{prefix}_expected_score.pkl")
    prediction_frame(score_support[probabilities.argmax(axis=-1)], split).to_pickle(run_dir / f"{prefix}_hard_score.pkl")
    bundle["cost"].to_pickle(run_dir / f"{prefix}_dollar_cost.pkl")


prediction_manifest = []
for position, ((method, feature, split), bundle) in enumerate(prediction_bundles.items()):
    prefix = f"pred_{position:02d}_{split}"
    save_prediction_bundle(prefix, bundle, split)
    prediction_manifest.append({
        "prefix": prefix, "method": method, "embedding": feature, "split": split,
        "k": DISPLAY_K if method == "kNN" else None,
    })
pd.DataFrame(prediction_manifest).to_csv(run_dir / "predictions_manifest.csv", index=False)

knn_prediction_manifest = []
for feature, split_predictions in knn_class_predictions.items():
    for split, all_k in split_predictions.items():
        for k, bundle in all_k.items():
            prefix = f"knn_{feature}_k{k}_{split}"
            save_prediction_bundle(prefix, bundle, split)
            knn_prediction_manifest.append({
                "prefix": prefix, "embedding": feature, "split": split, "k": k,
            })
pd.DataFrame(knn_prediction_manifest).to_csv(run_dir / "knn_predictions_manifest.csv", index=False)

linear_cv_table.to_csv(run_dir / "linear_cv_folds.csv", index=False)
linear_cv_summary.to_csv(run_dir / "linear_cv_summary.csv", index=False)
linear_fit_table.to_csv(run_dir / "linear_final_fit.csv", index=False)
joblib.dump(linear_models, run_dir / "linear_models.joblib")
joblib.dump(xgb_models, run_dir / "xgb_models.joblib")

manifest = {
    "mode": RUN_MODE, "seed": SEED,
    "model_names": model_names, "score_support": score_support.tolist(),
    "label_mapping": "nearest fixed anchor; exact midpoint uses lower anchor",
    "numerical_score_metrics_target": "original recorded score",
    "training_cost_means": cost_means.tolist(),
    "selected_linear_C": selected_linear_C, "cost_ridge_alpha": COST_RIDGE_ALPHA,
    "xgb_parameters": xgb_params, "k_values": K_VALUES, "display_k": DISPLAY_K,
    "rows": {split: len(tables[split]["questions"]) for split in SPLITS},
    "test_used": False, "calibration_fitted": False,
    "versions": {"sklearn": sklearn.__version__, "xgboost": xgboost.__version__,
                 "numpy": np.__version__, "pandas": pd.__version__},
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
(run_dir / "experiment_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Completed mode:", RUN_MODE)
print("Fixed score classes:", score_support.tolist())
print("Experiment outputs:", run_dir)
if RUN_MODE == "smoke":
    print("Smoke validates execution. Use full mode for performance comparisons.")


Completed mode: full
Fixed score classes: [0.0, 0.25, 0.5, 0.75, 1.0]
Experiment outputs: C:\Users\johnh\MachineLearning\project_erdos\2_Router_Training\experiment_outputs\score_classification_5class\full_seed42_20261013T080041_062763Z


## 9. Read the results

- Start with accuracy, balanced accuracy and NLL. A high accuracy combined with
  low balanced accuracy suggests weak performance on less frequent classes.
- Compare hard and expected outputs using raw-score MSE, MAE and WAPE. The
  outputs answer different questions: most likely category and average score.
- Compare TF-IDF and E5 using the same validation questions and metric definitions.
  Inspect the per-LLM CSV files when an average hides a weak candidate model.
- Use the training-frequency baseline to assess what the question features add.
- Verify `mode`, row counts and the class mapping in the saved manifest before
  interpreting an experiment. Smoke runs only check execution.

Further model selection and repeated validation comparisons can introduce
optimism. Reserve the untouched test split for the final selected experiment.


### Observations from the completed full run

This run uses all **21,900 train**, **3,648 calibration** and **3,649 validation**
questions. Training-only cross-validation selected **C = 1** for both feature
sets. All logistic fits converged. XGBoost used **100 boosting rounds** for every
score classifier and price model.

Five-class accuracy is about **68.0%?69.5%** across the six classifier/feature
combinations, compared with **53.8%** for the training-frequency baseline.
Hard class has lower raw-score MAE; expected score has lower raw-score MSE in
every combination. Read these metrics together because they measure different
errors. The two outputs share each method's price predictions.

**Execution note:** independent fits were computed in parallel and saved as
checkpoints during this run. Notebook loops then loaded the completed fits.
Their printed seconds include bulk precomputation or checkpoint loading; use
`fit_telemetry.csv` for the individual fit durations. Saved prediction tensors
have five class columns, and all requested kNN neighbor counts were exported.
